# VapourSynth DPIR 降噪 + 动画超分辨率（T4 × 2）

先用 DPIR `drunet_color` 对原尺寸画面轻度降噪，再将降噪结果送入两条放大分支：主体线条使用 `realesr-animevideov3`，平面与色度使用 NNEDI3，由主轮廓软遮罩融合。融合后在 16-bit YUV444 中进行两阶段 deband，并加入每帧变化的 Gaussian 颗粒，最后转换为 8-bit HEVC。默认先从任意起点处理 10 秒；确认效果和速度后再把 `TEST_SECONDS` 改为 `0`。

In [ ]:
!if [ -d /kaggle/working/Real-ESRGAN/.git ]; then git -C /kaggle/working/Real-ESRGAN pull --ff-only; else git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN; fi

In [ ]:
from pathlib import Path
RUNNER = Path("/kaggle/working/Real-ESRGAN/realesrgan_mask.py")
RUNNER_SOURCE = RUNNER.read_text(encoding="utf-8")
assert 'PIPELINE_VERSION = "dpir-deband-v1"' in RUNNER_SOURCE, "GitHub 上的 realesrgan_mask.py 仍是旧版；请先提交并推送本地 deband 改动，然后重新启动 Kaggle Session"
print("pipeline: dpir-deband-v1")

!python /kaggle/working/Real-ESRGAN/realesrgan_mask.py --version
!python --version
!nvidia-smi -L
!ffmpeg -version | head -n 1

In [ ]:
!pip install -q uv
!env -u UV_SYSTEM_PYTHON UV_LINK_MODE=copy uv venv --python 3.12 --seed /kaggle/working/vsenv
!/kaggle/working/vsenv/bin/python -m pip install -q -r /kaggle/working/Real-ESRGAN/requirements_mask.txt --extra-index-url https://jaded-encoding-thaumaturgy.github.io/vs-wheels/simple
!/kaggle/working/vsenv/bin/python -m pip install -q vapoursynth-mlrt-trt==16.0 || echo "TensorRT 后端安装失败，将使用 ORT CUDA"
!/kaggle/working/vsenv/bin/vapoursynth config
!/kaggle/working/vsenv/bin/vapoursynth check-env
!/kaggle/working/vsenv/bin/python -c "import vapoursynth as vs; print('VapourSynth:', vs.__version__); print('plugin_dir:', vs.get_plugin_dir())"
!/kaggle/working/vsenv/bin/python -c "import vapoursynth as vs; assert hasattr(vs.core, 'znedi3') or hasattr(vs.core, 'nnedi3'), 'NNEDI3 plugin not loaded'; print('NNEDI3: OK')"
!/kaggle/working/vsenv/bin/python -c "import vapoursynth as vs; from vsdeband import f3k_deband; assert hasattr(vs.core, 'vszip'), 'vszip plugin not loaded'; print('two-stage deband/dynamic grain: OK')"
!/kaggle/working/vsenv/bin/vspipe --version

In [ ]:
VS_PYTHON = "/kaggle/working/vsenv/bin/python"
VSPIPE = "/kaggle/working/vsenv/bin/vspipe"
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime2/cm_4.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan-mask.mp4"

# 倍率按宽高计算：720p → 2160p 是 3×；1080p → 2160p 是 2×
SCALE = 2
FPS = "source"                   # source / 23.976 / 24000/1001 / 60
START_TIME = 3 * 60 + 15                    # 任意起点，单位秒
TEST_SECONDS = 10                # 0 = 处理到视频末尾

# 当前 Kaggle pip 环境缺少 vs-mlrt 建引擎所需的 trtexec，直接用 ORT 可省去失败探测时间
BACKEND = "ort_cuda"            # trtexec 可用后再改为 auto / trt
GPU_IDS = "0,1"                # 两个独立进程，各自常驻一套模型并处理连续半段
FP16 = True
TILE_SIZE = 384                  # 更少的小图块调用；OOM 时依次改为 256、192、128
OVERLAP = 24
NUM_STREAMS = 2                   # T4 双流提高卷积吞吐；不稳定/OOM 时改回 1
BATCH_SIZE = 1                   # 可尝试 2 提高吞吐；OOM 或波动时保持 1
PREFER_NHWC = True               # ORT 卷积布局优化，不改变模型或遮罩
WORKSPACE_GIB = 4
CUDA_GRAPH = False               # ORT 保持关闭；TensorRT 后端可重新开启
CORE_THREADS = 2                 # 两个并行进程各 2 线程，避免 NNEDI3 争抢 Kaggle CPU
CACHE_MIB = 2048
DPIR_STRENGTH = 1.2               # 轻度降噪，尽量保留细线和纹理
DPIR_TILE_SIZE = 256              # 必须为 8 的倍数；OOM 时改为 192、128
DPIR_OVERLAP = 16                 # DPIR 官方默认重叠；不要为提速而设为 0

# 融合后的 16-bit YUV444 后处理：两阶段 deband + 高质量 Gaussian 动噪
DEBAND = True
DEBAND1_RADIUS = 24               # 第一遍处理较宽的色带
DEBAND1_THRESHOLD = 64.0
DEBAND2_RADIUS = 12               # 第二遍轻度精修
DEBAND2_THRESHOLD = 32.0
GRAIN_LUMA = 32.0                # 动噪强度；越高体积越大
GRAIN_CHROMA = 16.0
GRAIN_SEED = 333                 # 固定种子保证可复现，但每帧颗粒仍会变化
NNEDI3_NSIZE = 0                 # 官方建议放大使用 0 或 4；0 更锐利
NNEDI3_NNS = 2                   # 64 neurons；越高越慢，质量差异通常较小
NNEDI3_QUAL = 2                  # 两组预测融合，官方建议图像放大使用 2
NNEDI3_PSCRN = 2                 # 新版快速预筛选
INPUT_WIDTH = 0
INPUT_HEIGHT = 0

# 主线条遮罩：强预滤波 + 空间连续性门控，避免把颗粒/噪点交给 Real-ESRGAN
MASK_LOW = 0.040
MASK_HIGH = 0.140
MASK_DARK_BOOST = 0.5
MASK_STRENGTH = 1.0
MASK_INFLATE = 1
MASK_FEATHER = 1
MASK_PREFILTER = 2
MASK_SUPPORT_LOW = 0.012
MASK_SUPPORT_HIGH = 0.035
MASK_SUPPORT_RADIUS = 2

# 质量优先的 8-bit 硬件 HEVC；运行前会按真实输出尺寸测试编码器
VIDEO_CODEC = "hevc_nvenc"       # T4 NVENC；不可用时直接报错，不静默更换编码器
CQ = 14
NVENC_PRESET = "p7"
ENCODE_GPU = 0
CRF = 14
PRESET = "slow"
AUDIO_CODEC = "aac"             # 任意起点/10秒测试用 aac；完整视频从0开始可用 copy
AUDIO_BITRATE = "192k"
PROGRESS_INTERVAL = 60

In [ ]:
FP16_FLAG = "--fp16" if FP16 else "--no-fp16"
CUDA_GRAPH_FLAG = "--cuda-graph" if CUDA_GRAPH else "--no-cuda-graph"
NHWC_FLAG = "--prefer-nhwc" if PREFER_NHWC else "--no-prefer-nhwc"
DEBAND_FLAG = "--deband" if DEBAND else "--no-deband"

!{VS_PYTHON} /kaggle/working/Real-ESRGAN/realesrgan_mask.py \
  --input "{INPUT_VIDEO}" \
  --output "{OUTPUT_VIDEO}" \
  --vpy-script /kaggle/working/Real-ESRGAN/realesrgan_mask.vpy \
  --work-dir /kaggle/working/realesrgan-mask-assets \
  --engine-dir /kaggle/working/realesrgan-mask-engines \
  --scale {SCALE} \
  --fps "{FPS}" \
  --start-time {START_TIME} \
  --test-seconds {TEST_SECONDS} \
  --input-width {INPUT_WIDTH} \
  --input-height {INPUT_HEIGHT} \
  --backend "{BACKEND}" \
  --gpu-ids "{GPU_IDS}" \
  {FP16_FLAG} \
  --tile-size {TILE_SIZE} \
  --overlap {OVERLAP} \
  --num-streams {NUM_STREAMS} \
  --batch-size {BATCH_SIZE} \
  {NHWC_FLAG} \
  --workspace-gib {WORKSPACE_GIB} \
  {CUDA_GRAPH_FLAG} \
  --core-threads {CORE_THREADS} \
  --cache-mib {CACHE_MIB} \
  --dpir-strength {DPIR_STRENGTH} \
  --dpir-tile-size {DPIR_TILE_SIZE} \
  --dpir-overlap {DPIR_OVERLAP} \
  {DEBAND_FLAG} \
  --deband1-radius {DEBAND1_RADIUS} \
  --deband1-threshold {DEBAND1_THRESHOLD} \
  --deband2-radius {DEBAND2_RADIUS} \
  --deband2-threshold {DEBAND2_THRESHOLD} \
  --grain-luma {GRAIN_LUMA} \
  --grain-chroma {GRAIN_CHROMA} \
  --grain-seed {GRAIN_SEED} \
  --nnedi3-nsize {NNEDI3_NSIZE} \
  --nnedi3-nns {NNEDI3_NNS} \
  --nnedi3-qual {NNEDI3_QUAL} \
  --nnedi3-pscrn {NNEDI3_PSCRN} \
  --mask-low {MASK_LOW} \
  --mask-high {MASK_HIGH} \
  --mask-dark-boost {MASK_DARK_BOOST} \
  --mask-strength {MASK_STRENGTH} \
  --mask-inflate {MASK_INFLATE} \
  --mask-feather {MASK_FEATHER} \
  --mask-prefilter {MASK_PREFILTER} \
  --mask-support-low {MASK_SUPPORT_LOW} \
  --mask-support-high {MASK_SUPPORT_HIGH} \
  --mask-support-radius {MASK_SUPPORT_RADIUS} \
  --video-codec "{VIDEO_CODEC}" \
  --cq {CQ} \
  --nvenc-preset "{NVENC_PRESET}" \
  --encode-gpu {ENCODE_GPU} \
  --crf {CRF} \
  --preset "{PRESET}" \
  --audio-codec "{AUDIO_CODEC}" \
  --audio-bitrate "{AUDIO_BITRATE}" \
  --progress-interval {PROGRESS_INTERVAL} \
  --vspipe-bin "{VSPIPE}" \
  --ffmpeg-bin ffmpeg \
  --ffprobe-bin ffprobe

## OOM 降级顺序

1. 若把 `BATCH_SIZE` 调成了 2，先恢复为 `1`；
2. 把 `NUM_STREAMS` 从 2 改回 `1`；
3. 把 `DPIR_TILE_SIZE` 从 256 改为 `192`，再改为 `128`（`DPIR_OVERLAP` 保持 16）；
4. 把 `TILE_SIZE` 从 384 改为 `256`（`OVERLAP` 保持 24）；
5. 再改为 `TILE_SIZE = 192`、`OVERLAP = 20`，最后 `128/16`；
6. `CUDA_GRAPH = False`、`CACHE_MIB = 1024`、`WORKSPACE_GIB = 2`；
7. 保持 `GPU_IDS = "0,1"`；每张卡运行一个独立进程。不要降低 `DPIR_STRENGTH`、提高 `CQ/CRF` 或削弱遮罩来解决 OOM。

TensorRT 第一次运行需要建立并缓存引擎，首次启动时间明显更长；同一模型、tile、batch 和 T4 环境的后续运行会复用缓存。

`DPIR_STRENGTH = 1.2` 只做轻度预降噪。两阶段 deband 在放大与融合后执行：阈值越高越容易消除色带，也越可能抹平细节；半径影响色带检测范围。Gaussian 颗粒是逐帧变化的，`GRAIN_LUMA/CHROMA` 越高，越能掩盖残留色带，但 HEVC 输出体积也会明显增加；若体积过大，优先降为 `24/12`，不要提高 `CQ` 来抵消颗粒。遮罩参数决定 Real-ESRGAN 的参与范围，`NNEDI3_QUAL = 2` 是质量优先设置。tile 大小本身不应改变效果，但两种 overlap 都不要低于推荐值。`FPS` 高于源帧率只会复制已增强帧，不是补帧。